# DASH5 DAS Data Store Demo

This notebook demonstrates how to use the DASH5DataStore to read DAS (Distributed Acoustic Sensing) data stored in HDF5 format.

## Features

- Load DAS data from HDF5 files using glob patterns
- Read specific channels and time spans
- Convert data to ObsPy streams for further analysis
- Discover available data files

In [ ]:
import os
import logging
import numpy as np
from datetime import datetime, timezone
from datetimerange import DateTimeRange

# Set up logging to see info messages
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("noisepy_das.io")
logger.setLevel(logging.INFO)

In [ ]:
# Import our DAS data store
from noisepy_das.io import DASH5DataStore

## 1. Initialize the Data Store

First, we'll create a DASH5DataStore instance. You'll need to specify:
- `path`: Directory containing your H5 files
- `sampling_rate`: Sampling rate of your data (Hz)
- `channel_numbers`: List of channel indices to read
- `file_naming`: Pattern for parsing filenames (strptime format)

In [ ]:
# Example configuration - adjust these paths and parameters for your data
DAS_DATA_PATH = "../examples/sample_data/"  # Path to your H5 files
SAMPLING_RATE = 100  # Hz
CHANNELS = [0, 1, 2, 3, 4]  # Channel numbers to read
FILE_PATTERN = "%Y-%m-%d-%H-%M-%S.h5"  # Filename pattern

# Create the data store
raw_store = DASH5DataStore(
    path=DAS_DATA_PATH,
    sampling_rate=SAMPLING_RATE,
    channel_numbers=CHANNELS,
    file_naming=FILE_PATTERN,
    array_name="DAS",
    date_range=None  # Load all available data
)

print(f"Data store initialized for path: {DAS_DATA_PATH}")
print(f"Filesystem type: {type(raw_store.fs)}")

## 2. Discover Available Files

Use glob patterns to discover what H5 files are available in your data directory.

In [ ]:
# Discover all H5 files
h5_files = raw_store.discover_files("*.h5")
print(f"Found {len(h5_files)} H5 files:")
for i, file in enumerate(h5_files[:10]):  # Show first 10
    print(f"  {i+1:2d}: {os.path.basename(file)}")
if len(h5_files) > 10:
    print(f"     ... and {len(h5_files)-10} more files")

In [ ]:
# You can also use more specific patterns
# For example, files from a specific date
daily_files = raw_store.discover_files("2023-01-15-*.h5")
print(f"Found {len(daily_files)} files for 2023-01-15")

# Or files from a specific hour
hourly_files = raw_store.discover_files("2023-01-15-14-*.h5")
print(f"Found {len(hourly_files)} files for 2023-01-15 14:xx")

## 3. Examine Available Time Spans and Channels

In [ ]:
# Get available time spans
timespans = raw_store.get_timespans()
print(f"Available time spans: {len(timespans)}")
if len(timespans) > 0:
    print(f"First timespan: {timespans[0]}")
    print(f"Last timespan: {timespans[-1]}")
    
    # Get channels for the first timespan
    channels = raw_store.get_channels(timespans[0])
    print(f"\nChannels available for {timespans[0]}:")
    for i, channel in enumerate(channels[:5]):  # Show first 5
        print(f"  {i+1}: {channel}")
    if len(channels) > 5:
        print(f"     ... and {len(channels)-5} more channels")

## 4. Read Data for Specific Channel and Time

Now let's read actual data from a specific channel and time span.

In [ ]:
if len(timespans) > 0 and len(channels) > 0:
    # Read data for the first channel and timespan
    timespan = timespans[0]
    channel = channels[0]
    
    print(f"Reading data for channel {channel} at time {timespan}")
    data = raw_store.read_data(timespan, channel)
    
    print(f"Data shape: {data.data.shape}")
    print(f"Sampling rate: {data.sampling_rate} Hz")
    print(f"Start time: {datetime.fromtimestamp(data.start_timestamp, tz=timezone.utc)}")
    print(f"Data type: {data.data.dtype}")
    print(f"Data range: {data.data.min():.2f} to {data.data.max():.2f}")
    
    # Access the ObsPy stream
    print(f"\nObsPy Stream info:")
    print(data.stream)
else:
    print("No data available to read - check your data path and file format")

## 5. Read Multiple Channels

Let's read data from multiple channels for analysis.

In [ ]:
if len(timespans) > 0 and len(channels) > 1:
    timespan = timespans[0]
    
    # Read data from first 3 channels
    multi_channel_data = []
    for i, channel in enumerate(channels[:3]):
        print(f"Reading channel {i+1}/3: {channel}")
        data = raw_store.read_data(timespan, channel)
        multi_channel_data.append(data)
    
    print(f"\nLoaded {len(multi_channel_data)} channels")
    
    # Create a simple plot if matplotlib is available
    try:
        import matplotlib.pyplot as plt
        
        fig, axes = plt.subplots(len(multi_channel_data), 1, figsize=(12, 8))
        if len(multi_channel_data) == 1:
            axes = [axes]
            
        for i, data in enumerate(multi_channel_data):
            time_axis = np.arange(len(data.data)) / data.sampling_rate
            axes[i].plot(time_axis, data.data)
            axes[i].set_title(f"Channel {channels[i]}")
            axes[i].set_xlabel("Time (s)")
            axes[i].set_ylabel("Amplitude")
            
        plt.tight_layout()
        plt.show()
        
    except ImportError:
        print("Matplotlib not available - skipping plot")
        
else:
    print("Not enough data available for multi-channel demo")

## 6. Working with Date Ranges

You can also create a data store with a specific date range to limit data loading.

In [ ]:
# Create a data store with a specific date range
start_time = datetime(2023, 1, 15, 14, 0, 0, tzinfo=timezone.utc)
end_time = datetime(2023, 1, 15, 15, 0, 0, tzinfo=timezone.utc)
date_range = DateTimeRange(start_time, end_time)

print(f"Creating data store for date range: {date_range}")

limited_store = DASH5DataStore(
    path=DAS_DATA_PATH,
    sampling_rate=SAMPLING_RATE,
    channel_numbers=CHANNELS,
    file_naming=FILE_PATTERN,
    array_name="DAS",
    date_range=date_range
)

limited_timespans = limited_store.get_timespans()
print(f"Available timespans in range: {len(limited_timespans)}")
for span in limited_timespans[:5]:  # Show first 5
    print(f"  {span}")

## 7. Advanced File Discovery

Demonstrate advanced file discovery patterns for different use cases.

In [ ]:
print("File discovery examples:")
print("\n1. All H5 files:")
all_files = raw_store.discover_files("*.h5")
print(f"   Found {len(all_files)} files")

print("\n2. Files from January 2023:")
january_files = raw_store.discover_files("2023-01-*.h5")
print(f"   Found {len(january_files)} files")

print("\n3. Files from 14:00 hour on any day:")
hour14_files = raw_store.discover_files("*-14-*.h5")
print(f"   Found {len(hour14_files)} files")

print("\n4. Files from first 5 minutes of any hour:")
first5min_files = raw_store.discover_files("*-0[0-4]-*.h5")
print(f"   Found {len(first5min_files)} files")

# Show some example filenames if available
if len(all_files) > 0:
    print("\nExample filenames:")
    for file in all_files[:3]:
        print(f"   {os.path.basename(file)}")

## Summary

This notebook demonstrated:

1. **Initializing** a DASH5DataStore for reading DAS data from HDF5 files
2. **Discovering** available data files using glob patterns
3. **Reading** data from specific channels and time periods
4. **Working** with ObsPy streams for further analysis
5. **Limiting** data loading to specific date ranges
6. **Advanced** file discovery patterns

### Next Steps

- Integrate with NoisePy for ambient noise cross-correlation analysis
- Process multiple time periods for temporal analysis
- Apply preprocessing filters and transformations
- Export data to other formats (ASDF, MiniSEED, etc.)

### Notes for Your Data

Make sure to adjust:
- `DAS_DATA_PATH`: Path to your actual H5 files
- `SAMPLING_RATE`: Your data's sampling rate
- `CHANNELS`: Channel numbers you want to analyze
- `FILE_PATTERN`: Filename pattern matching your files
- Date ranges and glob patterns as needed